In [0]:
from pyspark.sql.functions import (
    col,
    arrays_zip,
    explode,
    to_timestamp,
    current_timestamp,
)

BRONZE_TABLE = "bronze_weather_demo"
SILVER_TABLE = "silver_weather_demo"

assert spark.catalog.tableExists(BRONZE_TABLE), (
    "Run Notebook 03 on its demo table first."
)

bronze_df = spark.table(BRONZE_TABLE)

print("Bronze records:", bronze_df.count())

Bronze records: 1


In [0]:
hourly_arrays_df = bronze_df.select(
    "city",
    "country",
    "region",
    "source",
    "batch_id",
    "load_type",
    "raw_file_path",
    col("load_timestamp").alias("bronze_load_timestamp"),

    col("hourly.time").alias("time"),
    col("hourly.temperature_2m").alias("temperature_2m"),
    col("hourly.relative_humidity_2m").alias("relative_humidity_2m"),
    col("hourly.apparent_temperature").alias("apparent_temperature"),
    col("hourly.precipitation").alias("precipitation"),
    col("hourly.rain").alias("rain"),
    col("hourly.snowfall").alias("snowfall"),
    col("hourly.weather_code").alias("weather_code"),
    col("hourly.pressure_msl").alias("pressure_msl"),
    col("hourly.surface_pressure").alias("surface_pressure"),
    col("hourly.cloud_cover").alias("cloud_cover"),
    col("hourly.wind_speed_10m").alias("wind_speed_10m"),
    col("hourly.wind_direction_10m").alias("wind_direction_10m"),
)

zipped_df = hourly_arrays_df.withColumn(
    "weather_array",
    arrays_zip(
        "time",
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "pressure_msl",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m",
    ),
)

exploded_df = zipped_df.select(
    "city",
    "country",
    "region",
    "source",
    "batch_id",
    "load_type",
    "raw_file_path",
    "bronze_load_timestamp",
    explode("weather_array").alias("weather"),
)

print("Exploded hourly rows:", exploded_df.count())


Exploded hourly rows: 168


In [0]:
silver_df = exploded_df.select(
    "city",
    "country",
    "region",

    to_timestamp(
        col("weather.time"),
        "yyyy-MM-dd'T'HH:mm"
    ).alias("timestamp"),

    col("weather.temperature_2m").cast("double").alias("temperature_2m"),
    col("weather.relative_humidity_2m").cast("double").alias("relative_humidity_2m"),
    col("weather.apparent_temperature").cast("double").alias("apparent_temperature"),
    col("weather.precipitation").cast("double").alias("precipitation"),
    col("weather.rain").cast("double").alias("rain"),
    col("weather.snowfall").cast("double").alias("snowfall"),
    col("weather.weather_code").cast("int").alias("weather_code"),
    col("weather.pressure_msl").cast("double").alias("pressure_msl"),
    col("weather.surface_pressure").cast("double").alias("surface_pressure"),
    col("weather.cloud_cover").cast("double").alias("cloud_cover"),
    col("weather.wind_speed_10m").cast("double").alias("wind_speed_10m"),
    col("weather.wind_direction_10m").cast("double").alias("wind_direction_10m"),

    "source",
    "batch_id",
    "load_type",
    "raw_file_path",
    "bronze_load_timestamp",
    current_timestamp().alias("load_timestamp"),
)

silver_df.printSchema()
display(silver_df.limit(10))


root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- source: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- raw_file_path: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)



city,country,region,timestamp,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,source,batch_id,load_type,raw_file_path,bronze_load_timestamp,load_timestamp
London,United Kingdom,Western Europe,2024-01-01T00:00:00.000Z,8.2,81.0,2.7,0.0,0.0,0.0,2,995.3,993.4,52.0,29.6,246.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T01:00:00.000Z,7.9,82.0,2.5,0.0,0.0,0.0,2,996.1,994.2,67.0,28.5,249.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T02:00:00.000Z,7.9,80.0,2.5,0.0,0.0,0.0,2,997.1,995.2,57.0,27.6,250.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T03:00:00.000Z,7.9,80.0,2.5,0.0,0.0,0.0,1,998.1,996.2,41.0,27.6,253.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T04:00:00.000Z,7.7,79.0,2.4,0.0,0.0,0.0,1,998.8,996.9,20.0,26.6,254.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T05:00:00.000Z,7.5,80.0,2.4,0.0,0.0,0.0,1,999.7,997.8,35.0,25.2,256.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T06:00:00.000Z,7.3,79.0,2.4,0.0,0.0,0.0,1,1000.7,998.8,46.0,23.7,256.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T07:00:00.000Z,7.2,79.0,2.4,0.0,0.0,0.0,3,1001.8,999.8,83.0,22.4,255.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T08:00:00.000Z,6.8,79.0,2.3,0.0,0.0,0.0,3,1002.9,1000.9,94.0,20.7,253.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z
London,United Kingdom,Western Europe,2024-01-01T09:00:00.000Z,6.7,79.0,2.3,0.0,0.0,0.0,3,1003.6,1001.6,96.0,19.1,248.0,Open-Meteo ERA5,London_2024-01-01_2024-01-07,TEST,/Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json,2026-10-10T22:49:56.000Z,2026-10-10T23:12:27.614Z


In [0]:
total_rows = silver_df.count()

distinct_keys = (
    silver_df
    .select("city", "timestamp")
    .distinct()
    .count()
)

duplicate_count = total_rows - distinct_keys

null_timestamp_count = (
    silver_df
    .filter(col("timestamp").isNull())
    .count()
)

invalid_humidity_count = (
    silver_df
    .filter(
        (col("relative_humidity_2m") < 0) |
        (col("relative_humidity_2m") > 100)
    )
    .count()
)

print("===== SILVER VALIDATION =====")
print("Total rows:", total_rows)
print("Distinct city + timestamp:", distinct_keys)
print("Duplicate city + timestamp:", duplicate_count)
print("Null timestamps:", null_timestamp_count)
print("Invalid humidity rows:", invalid_humidity_count)

assert duplicate_count == 0, "Duplicate city + timestamp records found"
assert null_timestamp_count == 0, "Null timestamps found"
assert invalid_humidity_count == 0, "Humidity outside 0–100 found"


===== SILVER VALIDATION =====
Total rows: 168
Distinct city + timestamp: 168
Duplicate city + timestamp: 0
Null timestamps: 0
Invalid humidity rows: 0


In [0]:

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

silver_check = spark.table(SILVER_TABLE)

print("Persisted Silver rows:", silver_check.count())

silver_check.select(
    "city",
    "timestamp",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "load_timestamp",
).show(10, truncate=False)

print("Bronze → Silver completed successfully.")


Persisted Silver rows: 168
+------+-------------------+--------------+--------------------+-------------+--------------+--------------------------+
|city  |timestamp          |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|load_timestamp            |
+------+-------------------+--------------+--------------------+-------------+--------------+--------------------------+
|London|2024-01-01 00:00:00|8.2           |81.0                |0.0          |29.6          |2026-10-10 23:12:41.119793|
|London|2024-01-01 01:00:00|7.9           |82.0                |0.0          |28.5          |2026-10-10 23:12:41.119793|
|London|2024-01-01 02:00:00|7.9           |80.0                |0.0          |27.6          |2026-10-10 23:12:41.119793|
|London|2024-01-01 03:00:00|7.9           |80.0                |0.0          |27.6          |2026-10-10 23:12:41.119793|
|London|2024-01-01 04:00:00|7.7           |79.0                |0.0          |26.6          |2026-10-10 23:12:41.119793|
|Lond